# Exercice, Appliquer un extrait "update" KBO sur bronze et silver

Vous avez déjà un dossier d'extrait journalier téléchargé, par exemple
`KboOpenData_0432_2026_07_26_Update/`, qui contient :

- `meta.csv` : numéro d'extrait (`ExtractNumber`), type (`ExtractType`),
  date du snapshot (`SnapshotDate`)
- pour chaque entité (`enterprise`, `establishment`, `branch`,
  `denomination`, `address`, `contact`, `activity`) : un fichier
  `{entite}_insert.csv` et un fichier `{entite}_delete.csv`

**Important** pour`enterprise`/`establishment`/`branch`, chaque ligne d'insert a une clé
unique (`EnterpriseNumber`/`EstablishmentNumber`/`Id`)

Mais pour `denomination`/`address`/`contact`/`activity`, le fichier delete
ne liste QUE `EntityNumber` (pas de clé de ligne précise), ça veut dire
que dès qu'UN SEUL élément change pour une entité (ex: une seule activité
NACE ajoutée), TOUTES les lignes de cette entité dans cette table sont
supprimées puis réinsérées en entier.

vous allez :
1. Lire `meta.csv` et les fichiers insert/delete (assurez vous de pas appliquer la meme mise a jour deux fois)
2. Appliquer la mise à jour sur bronze 
3. Ne reconstruire `entreprise`/`entreprise_silver` QUE pour les
   entreprises réellement affectées par ce lot (pas un rebuild complet)
4. Éviter de rejouer deux fois le même extrait (meta.csv contient un snapshot)


## 1. Lire `meta.csv` et les fichiers insert/delete


In [17]:

from pymongo import MongoClient

# Adjust the connection URI and database name to match your environment
client = MongoClient("mongodb://localhost:27017/")
db = client["kbo_database"]


In [18]:
from pathlib import Path
import csv
from datetime import datetime
from pymongo import DeleteOne, DeleteMany, InsertOne, ReplaceOne
from pymongo.errors import BulkWriteError

# Assurez-vous que l'index unique existe sur la collection Silver
db.kbo_entreprise_silver.create_index("EnterpriseNumber", unique=True)
print("Index unique créé sur kbo_entreprise_silver.EnterpriseNumber")

Index unique créé sur kbo_entreprise_silver.EnterpriseNumber


In [19]:
from pathlib import Path
import csv
from datetime import datetime
from pymongo import DeleteOne, DeleteMany, InsertOne, ReplaceOne
from pymongo.errors import BulkWriteError
# from pymongo import MongoClient
# client = MongoClient("mongodb://localhost:27017/")
# db = client["kbo_database"]

# 1. Trouver et trier tous les dossiers de mise à jour (0432, puis 0433, puis 0434)
base_path = Path(".")
update_dirs = sorted(base_path.glob("KboOpenData_*_Update"))

print(f"{len(update_dirs)} dossiers de mise à jour trouvés.\n")

for UPDATE_DIR in update_dirs:
    print(f"{'='*50}")
    print(f"Traitement du dossier : {UPDATE_DIR.name}")
    
    meta_path = UPDATE_DIR / "meta.csv"
    if not meta_path.exists():
        print(f"Ignoré : Aucun fichier meta.csv trouvé.")
        continue

    # -------------------------------------------------------------------------
    # ÉTAPE 1 : Lire meta.csv (Format Clé / Valeur)
    # -------------------------------------------------------------------------
    meta_data = {}
    with open(meta_path, encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            meta_data[row["Variable"]] = row["Value"]

    extract_number = meta_data["ExtractNumber"]

    # Vérifier si l'extrait a déjà été appliqué
    if db.kbo_update_log.find_one({"ExtractNumber": extract_number}):
        print(f"-> Extrait {extract_number} déjà appliqué. Passage au suivant.")
        continue
        
    print(f"-> Application de l'extrait : {extract_number}...")

    # -------------------------------------------------------------------------
    # ÉTAPE 2 : Calcul des entreprises affectées (AVANT les suppressions)
    # -------------------------------------------------------------------------
    affected_enterprises = set()
    entities = ["enterprise", "establishment", "branch", "denomination", "address", "contact", "activity"]

    if (UPDATE_DIR / "establishment_delete.csv").exists():
        with open(UPDATE_DIR / "establishment_delete.csv", encoding="utf-8-sig") as f:
            for row in csv.DictReader(f):
                doc = db.kbo_establishment.find_one({"EstablishmentNumber": row.get("EstablishmentNumber")}, {"EnterpriseNumber": 1})
                if doc and "EnterpriseNumber" in doc:
                    affected_enterprises.add(doc["EnterpriseNumber"])

    if (UPDATE_DIR / "branch_delete.csv").exists():
        with open(UPDATE_DIR / "branch_delete.csv", encoding="utf-8-sig") as f:
            for row in csv.DictReader(f):
                doc = db.kbo_branch.find_one({"Id": row.get("Id")}, {"EnterpriseNumber": 1})
                if doc and "EnterpriseNumber" in doc:
                    affected_enterprises.add(doc["EnterpriseNumber"])

    for ent in entities:
        for action in ["insert", "delete"]:
            filepath = UPDATE_DIR / f"{ent}_{action}.csv"
            if not filepath.exists(): 
                continue
            with open(filepath, encoding="utf-8-sig") as f:
                for row in csv.DictReader(f):
                    if "EnterpriseNumber" in row:
                        affected_enterprises.add(row["EnterpriseNumber"])
                    elif "EntityNumber" in row:
                        affected_enterprises.add(row["EntityNumber"])

    print(f"   - {len(affected_enterprises)} entreprises à reconstruire identifiées.")

    # -------------------------------------------------------------------------
    # ÉTAPE 3 : Application sur la couche Bronze
    # -------------------------------------------------------------------------
    def process_entity_bronze(entity_name, key_field, is_multi_delete=False):
        col = db[f"kbo_{entity_name}"]
        
        # Deletes
        delete_file = UPDATE_DIR / f"{entity_name}_delete.csv"
        if delete_file.exists():
            requests = []
            with open(delete_file, encoding="utf-8-sig") as f:
                for row in csv.DictReader(f):
                    if is_multi_delete:
                        requests.append(DeleteMany({"EntityNumber": row["EntityNumber"]}))
                    else:
                        requests.append(DeleteOne({key_field: row[key_field]}))
            if requests:
                col.bulk_write(requests)

        # Inserts
        insert_file = UPDATE_DIR / f"{entity_name}_insert.csv"
        if insert_file.exists():
            requests = []
            with open(insert_file, encoding="utf-8-sig") as f:
                for row in csv.DictReader(f):
                    if is_multi_delete:
                        requests.append(InsertOne(row))
                    else:
                        requests.append(ReplaceOne({key_field: row[key_field]}, row, upsert=True))
            if requests:
                try:
                    col.bulk_write(requests)
                except BulkWriteError as bwe:
                    print(f"Erreur sur {entity_name}:", bwe.details)

    process_entity_bronze("enterprise", "EnterpriseNumber")
    process_entity_bronze("establishment", "EstablishmentNumber")
    process_entity_bronze("branch", "Id")
    process_entity_bronze("denomination", "EntityNumber", is_multi_delete=True)
    process_entity_bronze("address", "EntityNumber", is_multi_delete=True)
    process_entity_bronze("contact", "EntityNumber", is_multi_delete=True)
    process_entity_bronze("activity", "EntityNumber", is_multi_delete=True)
    
    print("   - Couche Bronze mise à jour.")

    # -------------------------------------------------------------------------
    # ÉTAPE 4 : Propagation vers la couche Silver
    # -------------------------------------------------------------------------
    if affected_enterprises:
        pipeline = [
            {"$match": {"EnterpriseNumber": {"$in": list(affected_enterprises)}}},
            
            # --- VOS JOINTURES SILVER ICI ($lookup, etc.) ---
            {"$lookup": {"from": "kbo_denomination", "localField": "EnterpriseNumber", "foreignField": "EntityNumber", "as": "denominations"}},
            {"$lookup": {"from": "kbo_address", "localField": "EnterpriseNumber", "foreignField": "EntityNumber", "as": "addresses"}},
            {"$lookup": {"from": "kbo_activity", "localField": "EnterpriseNumber", "foreignField": "EntityNumber", "as": "activities"}},
            
            {"$merge": {
                "into": "kbo_entreprise_silver", 
                "on": "EnterpriseNumber", 
                "whenMatched": "replace", 
                "whenNotMatched": "insert"
            }}
        ]
        db.kbo_enterprise.aggregate(pipeline)
        print("   - Couche Silver mise à jour pour les entreprises affectées.")

    # -------------------------------------------------------------------------
    # ÉTAPE 5 : Log de l'extrait
    # -------------------------------------------------------------------------
    db.kbo_update_log.insert_one({
        "ExtractNumber": extract_number,
        "ExtractType": meta_data.get("ExtractType"),
        "SnapshotDate": meta_data.get("SnapshotDate"),
        "ExtractTimestamp": meta_data.get("ExtractTimestamp"),
        "AppliedAt": datetime.utcnow()
    })
    print(f"   - ✅ Extrait {extract_number} terminé et loggé avec succès.")

print(f"\n{'='*50}")
print("Tous les dossiers ont été traités avec succès !")

3 dossiers de mise à jour trouvés.

Traitement du dossier : KboOpenData_0432_2026_07_26_Update
-> Extrait 432 déjà appliqué. Passage au suivant.
Traitement du dossier : KboOpenData_0433_2026_07_27_Update
-> Extrait 433 déjà appliqué. Passage au suivant.
Traitement du dossier : KboOpenData_0434_2026_07_28_Update
-> Extrait 434 déjà appliqué. Passage au suivant.

Tous les dossiers ont été traités avec succès !


## 2. Appliquer la mise à jour sur bronze

Pour chacune des 7 entités, appliquez le insert/delete sur sa collection
brute (`kbo_enterprise`, `kbo_establishment`, `kbo_branch`,
`kbo_denomination`, `kbo_address`, `kbo_contact`, `kbo_activity`) :

In [20]:
# Fonction réutilisable pour appliquer les modifications sur la couche Bronze
def apply_bronze_updates(update_dir, entity_name, key_field, is_multi_delete=False):
    col = db[f"kbo_{entity_name}"]
    
    # 1. Traitement des suppressions (Deletes)
    delete_file = update_dir / f"{entity_name}_delete.csv"
    if delete_file.exists():
        requests = []
        with open(delete_file, encoding="utf-8-sig") as f:
            for row in csv.DictReader(f):
                if is_multi_delete:
                    requests.append(DeleteMany({"EntityNumber": row["EntityNumber"]}))
                else:
                    requests.append(DeleteOne({key_field: row[key_field]}))
        if requests:
            col.bulk_write(requests)

    # 2. Traitement des insertions (Inserts)
    insert_file = update_dir / f"{entity_name}_insert.csv"
    if insert_file.exists():
        requests = []
        with open(insert_file, encoding="utf-8-sig") as f:
            for row in csv.DictReader(f):
                if is_multi_delete:
                    requests.append(InsertOne(row))
                else:
                    requests.append(ReplaceOne({key_field: row[key_field]}, row, upsert=True))
        if requests:
            try:
                col.bulk_write(requests)
            except BulkWriteError as bwe:
                print(f"Erreur lors de l'insertion sur {entity_name}:", bwe.details)

print("Fonction de mise à jour de la couche Bronze définie.")

Fonction de mise à jour de la couche Bronze définie.


## 3. Propager vers `entreprise` et `entreprise_silver` !! SEULEMENT pour les entreprises affectées !!

Un rebuild complet de `entreprise_silver` (drop + réinsertion de toute la
base) serait du gâchis pour un lot qui ne touche que quelques centaines
d'entreprises sur des millions.

Attention à l'ORDRE : calculez l'ensemble affecté AVANT d'avoir appliqué les
deletes de l'étape précédente (sinon vous ne pourrez plus retrouver le
propriétaire d'un établissement/succursale déjà supprimé).


In [21]:
# Fonction réutilisable pour propager les modifications des entreprises affectées vers la couche Silver
def propagate_to_silver(affected_enterprises):
    if not affected_enterprises:
        print("Aucune entreprise affectée à mettre à jour dans Silver.")
        return

    # Pipeline d'agrégation ciblant STRICTEMENT les entreprises affectées
    pipeline = [
        {"$match": {"EnterpriseNumber": {"$in": list(affected_enterprises)}}},
        
        {"$lookup": {
            "from": "kbo_denomination", 
            "localField": "EnterpriseNumber", 
            "foreignField": "EntityNumber", 
            "as": "denominations"
        }},
        {"$lookup": {
            "from": "kbo_address", 
            "localField": "EnterpriseNumber", 
            "foreignField": "EntityNumber", 
            "as": "addresses"
        }},
        {"$lookup": {
            "from": "kbo_activity", 
            "localField": "EnterpriseNumber", 
            "foreignField": "EntityNumber", 
            "as": "activities"
        }},
        
        {"$merge": {
            "into": "kbo_entreprise_silver", 
            "on": "EnterpriseNumber", 
            "whenMatched": "replace", 
            "whenNotMatched": "insert"
        }}
    ]
    
    db.kbo_enterprise.aggregate(pipeline)
    print(f"Mise à jour Silver effectuée avec succès pour {len(affected_enterprises)} entreprises.")

## 4. Ne pas rejouer deux fois le même extrait

Créez une collection `kbo_update_log` qui enregistre, pour chaque extrait
appliqué avec succès : `extractNumber`, `extractType`, `snapshotDate`, et la
date/heure d'application.

In [22]:
# Vérification du contenu du journal des mises à jour (kbo_update_log)
print("=== Historique des mises à jour appliquées ===")
cursor = db.kbo_update_log.find().sort("ExtractNumber", 1)

for log in cursor:
    print(f"Extrait : {log.get('ExtractNumber')} | Type : {log.get('ExtractType')} | Date Snapshot : {log.get('SnapshotDate')} | Appliqué le : {log.get('AppliedAt')}")

=== Historique des mises à jour appliquées ===
Extrait : 432 | Type : update | Date Snapshot : 25-07-2026 | Appliqué le : 2026-07-29 09:20:51.693000
Extrait : 433 | Type : update | Date Snapshot : 26-07-2026 | Appliqué le : 2026-07-29 09:20:51.859000
Extrait : 434 | Type : update | Date Snapshot : 27-07-2026 | Appliqué le : 2026-07-29 09:20:54.043000
